In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Introduction
In this notebook I built the following AI Researcher Agents as instructed in the AI Agents Capestone Project overview to choose minimum of three cases learnt during the 5 Days AI Agents Intensive Course: Single AI Agent, Multi-AI Agent team, Sequential AI Agent team,Parallel AI Agent team(14 parallel AI Agents) then the Coordinator AI Agent,Aggregirator AI Agent, Reporter AI Agent making a total of 17 AI Agents team. i The finalise with Loop AI Agents team.

Lets start by geting the Google API key to get credential to do the project as instructed during the course. The following code will help the Gemini API key setup.

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

✅ Gemini API key setup complete.


Import ADK components

Let, import the specific components needed from the Agent Development Kit and 
the Generative AI library. This is to keep my  code organized and ensures I have access 
to the necessary building blocks.

In [3]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("✅ ADK components imported successfully.")

✅ ADK components imported successfully.


Configure Retry Options

Inorder not to encounter transient errors like rate limits or temporary service unavailability. Retry options automatically handle these failures by retrying the request with exponential backoff.

In [4]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

Why Multi-Agent Systems? + Your First Multi-Agent

The Problem: The "Do-It-All" Agent

Single agents can do a lot. But what happens when the task gets complex? A single "monolithic" agent that tries to do research, writing, editing, and fact-checking all at once becomes a problem. Its instruction prompt gets long and confusing. It's hard to debug (which part failed?), difficult to maintain, and often produces unreliable results.

The Solution: A Team of Specialists

Instead of one "do-it-all" agent, we can build a multi-agent system. This is a team of simple, specialized agents that collaborate, just like a real-world team. Each agent has one clear job (e.g., one agent only does research, another only writes). This makes them easier to build, easier to test, and much more powerful and reliable when working together.

To learn more, check out the documentation related to LLM agents in ADK.

Architecture: Single Agent vs Multi-Agent Team
Multi-agent Team

<img width="800" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/multi-agent-team.png" alt="Multi-agent Team" />

Blog Post Creation with Sequential Agents

Let's build a system with three specialized agents:

    Outline Agent - Creates a blog outline for a given topic
    Writer Agent - Writes a blog post
    Editor Agent - Edits a blog post draft for clarity and structure


In [5]:
# Research Agent: Its job is to use the google_search tool and present findings.
research_agent = Agent(
    name="ResearchAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a specialized research agent. Your only job is to use the
    google_search tool to find 2-3 pieces of relevant information on the given topic and present the findings with citations.""",
    tools=[google_search],
    output_key="research_findings",  # The result of this agent will be stored in the session state with this key.
)

print("✅ research_agent created.")

✅ research_agent created.


In [6]:
# Summarizer Agent: Its job is to summarize the text it receives.
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The instruction is modified to request a bulleted list for a clear output format.
    instruction="""Read the provided research findings: {research_findings}
Create a concise summary as a bulleted list with 3-5 key points.""",
    output_key="final_summary",
)

print("✅ summarizer_agent created.")

✅ summarizer_agent created.


In [7]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
)

print("✅ root_agent created.")

✅ root_agent created.


In [8]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "What is Algorand Blockchain?"
)


 ### Created new session: debug_session_id

User > What is Algorand Blockchain?


ResearchCoordinator > Algorand is a public blockchain platform that emphasizes security, scalability, and decentralization. It utilizes a unique Pure Proof-of-Stake (PPoS) consensus mechanism, which allows participants to validate transactions and propose blocks by staking ALGO tokens, ensuring decentralization and security without requiring significant computational power. The platform is designed for rapid transaction processing with near-instant finality and supports smart contracts for developing decentralized applications (dApps). Key features include low transaction costs and decentralized governance, where ALGO token holders can vote on network decisions. Algorand is a competitor to Ethereum and finds use cases in payments, decentralized finance (DeFi), and asset tokenization.


In [9]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "What is Model Context Protocol?"
)


 ### Created new session: debug_session_id

User > What is Model Context Protocol?


ResearchCoordinator > The Model Context Protocol (MCP) is an open-source framework that standardizes how AI systems, such as large language models (LLMs), integrate and share data with external tools, systems, and data sources. Introduced by Anthropic, it acts as a universal interface, allowing AI applications to connect to various data sources and tools, thus overcoming the limitations of LLMs having static knowledge and inability to access real-time data. MCP operates on a client-server architecture, simplifying the integration of AI applications with external systems and enabling AI agents to be more context-aware and take autonomous actions. Major AI providers like OpenAI and Google DeepMind have adopted this protocol.


Sequential Workflows - The Assembly Line

The Problem: Unpredictable Order

The previous multi-agent system worked, but it relied on a detailed instruction prompt to force the LLM to run steps in order. This can be unreliable. A complex LLM might decide to skip a step, run them in the wrong order, or get "stuck," making the process unpredictable.

The Solution: A Fixed Pipeline

When you need tasks to happen in a guaranteed, specific order, you can use a SequentialAgent. This agent acts like an assembly line, running each sub-agent in the exact order you list them. The output of one agent automatically becomes the input for the next, creating a predictable and reliable workflow.

Use Sequential when: Order matters, you need a linear pipeline, or each step builds on the previous one.

To learn more, check out the documentation related to sequential agents in ADK.

Architecture: Blog Post Creation Pipeline

<img width="1000" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/sequential-agent.png" alt="Sequential Agent" />

Example: Blog Post Creation with Sequential Agents

Let's build a system with three specialized agents:

    Outline Agent - Creates a blog outline for a given topic
    Writer Agent - Writes a blog post
    Editor Agent - Edits a blog post draft for clarity and structure


In [10]:
# Outline Agent: Creates the initial blog post outline.
outline_agent = Agent(
    name="OutlineAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Create a blog outline for the given topic with:
    1. A catchy headline
    2. An introduction hook
    3. 3-5 main sections with 2-3 bullet points for each
    4. A concluding thought""",
    output_key="blog_outline",  # The result of this agent will be stored in the session state with this key.
)

print("✅ outline_agent created.")

✅ outline_agent created.


In [11]:
# Writer Agent: Writes the full blog post based on the outline from the previous agent.
writer_agent = Agent(
    name="WriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # The `{blog_outline}` placeholder automatically injects the state value from the previous agent's output.
    instruction="""Following this outline strictly: {blog_outline}
    Write a brief, 200 to 300-word blog post with an engaging and informative tone.""",
    output_key="blog_draft",  # The result of this agent will be stored with this key.
)

print("✅ writer_agent created.")

✅ writer_agent created.


In [12]:
# Editor Agent: Edits and polishes the draft from the writer agent.
editor_agent = Agent(
    name="EditorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This agent receives the `{blog_draft}` from the writer agent's output.
    instruction="""Edit this draft: {blog_draft}
    Your task is to polish the text by fixing any grammatical errors, improving the flow and sentence structure, and enhancing overall clarity.""",
    output_key="final_blog",  # This is the final output of the entire pipeline.
)

print("✅ editor_agent created.")

✅ editor_agent created.


Then we bring the agents together under a root agent, or coordinator:

In [13]:
# Root Coordinator: Orchestrates the workflow by calling the sub-agents as tools.
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # This instruction tells the root agent HOW to use its tools (which are the other agents).
    instruction="""You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
1. First, you MUST call the `ResearchAgent` tool to find relevant information on the topic provided by the user.
2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
3. Finally, present the final summary clearly to the user as your response.""",
    # We wrap the sub-agents in `AgentTool` to make them callable tools for the root agent.
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)],
)

print("✅ root_agent created.")

✅ root_agent created.


Here we're using AgentTool to wrap the sub-agents to make them callable tools for the root agent. We'll explore AgentTool in-detail on Day 2.

Let's run the agent and ask it about a topic:


In [14]:
root_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent],
)

print("✅ Sequential Agent created.")

✅ Sequential Agent created.


In [15]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    "Write a blog post on how Model Context Protocol(MCP) can utilizes NFT's and Smart Contract to mitiget it's risks ?"
)


 ### Created new session: debug_session_id

User > Write a blog post on how Model Context Protocol(MCP) can utilizes NFT's and Smart Contract to mitiget it's risks ?
OutlineAgent > ## Outline: The Guardian of AI: How MCP Uses NFTs & Smart Contracts to Tame the Wild West of AI Risks

### Introduction Hook:

The AI revolution is here, but with immense power comes equally immense risk. From bias and errors to intellectual property theft and manipulation, the potential pitfalls of AI are a growing concern. What if there was a way to build trust and accountability directly into AI models? Enter the Model Context Protocol (MCP), a groundbreaking approach leveraging the unique capabilities of NFTs and smart contracts to mitigate these inherent risks.

### Main Sections:

**1. Understanding the Problem: The Unseen Risks in the AI Black Box**

*   **Lack of Transparency & Auditability:** Traditional AI models are often "black boxes," making it difficult to understand their decision-making proc

In [16]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug (
    """Write a blog post about How to use AI Agents most importantly ADK,blockchain 
    technology for example Algorand Blockchain technology(that is making use of it's NFT's
    Algorand Standard Asset,and/or Smart Contract) to streamline the process of 
    integrating tools and models,and address some of these technical and security challenges.
  runner  find better and most efficient ways to improve on the present Models Context Protocol (MCP).
    Consider some possible ways to mittigaget the RISKs, security concerns and exfiltarations of important organizational data and API's.
   Furthermore outline 10 bullet piont and later explain 10 ways that you could use to improve on the present MCP.
   """
   )


 ### Created new session: debug_session_id

User > Write a blog post about How to use AI Agents most importantly ADK,blockchain 
    technology for example Algorand Blockchain technology(that is making use of it's NFT's
    Algorand Standard Asset,and/or Smart Contract) to streamline the process of 
    integrating tools and models,and address some of these technical and security challenges.
  runner  find better and most efficient ways to improve on the present Models Context Protocol (MCP).
    Consider some possible ways to mittigaget the RISKs, security concerns and exfiltarations of important organizational data and API's.
   Furthermore outline 10 bullet piont and later explain 10 ways that you could use to improve on the present MCP.
   
OutlineAgent > ## Blog Outline: AI Agents, Algorand, and the Future of Streamlined Integration

**Catchy Headline:** Unleash the Power Duo: AI Agents & Algorand for Seamless Integration and Enhanced Security

**Introduction Hook:** Imagine a wo

In [17]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug (
    """Re-write the above blog post about How to use AI Agents most importantly ADK,blockchain 
    technology for example Algorand Blockchain technology(that is making use of it's NFT's
    Algorand Standard Asset,and/or Smart Contract) to streamline the process of 
    integrating tools and models,and address some of these technical and security challenges.
  runner  find better and most efficient ways to improve on the present Models Context Protocol (MCP).
    Consider some possible ways to mittigaget the RISKs, security concerns and exfiltarations of important organizational data and API's.
   Furthermore outline 10 bullet piont and later explain 10 ways that you could use to improve on the present MCP.

   considering the given outlines; 
   Problem Statement -- the problem you're trying to solve, and why you think it's an important or interesting problem to solve
Why agents? -- Why are agents the right solution to this problem
What you created -- What's the overall architecture?
Demo -- Show your solution
The Build -- How you created it, what tools or technologies you used.
If I had more time, this is what I'd do
   """
   )


 ### Created new session: debug_session_id

User > Re-write the above blog post about How to use AI Agents most importantly ADK,blockchain 
    technology for example Algorand Blockchain technology(that is making use of it's NFT's
    Algorand Standard Asset,and/or Smart Contract) to streamline the process of 
    integrating tools and models,and address some of these technical and security challenges.
  runner  find better and most efficient ways to improve on the present Models Context Protocol (MCP).
    Consider some possible ways to mittigaget the RISKs, security concerns and exfiltarations of important organizational data and API's.
   Furthermore outline 10 bullet piont and later explain 10 ways that you could use to improve on the present MCP.

   considering the given outlines; 
   Problem Statement -- the problem you're trying to solve, and why you think it's an important or interesting problem to solve
Why agents? -- Why are agents the right solution to this problem
What yo

Parallel Workflows - Independent Researchers

The Problem: The Bottleneck

The previous sequential agent is great, but it's an assembly line. Each step must wait for the previous one to finish. What if you have several tasks that are not dependent on each other? For example, researching three different topics. Running them in sequence would be slow and inefficient, creating a bottleneck where each task waits unnecessarily.

The Solution: Concurrent Execution

When you have independent tasks, you can run them all at the same time using a ParallelAgent. This agent executes all of its sub-agents concurrently, dramatically speeding up the workflow. Once all parallel tasks are complete, you can then pass their combined results to a final 'aggregator' step.

Use Parallel when: Tasks are independent, speed matters, and you can execute concurrently.

To learn more, check out the documentation related to parallel agents in ADK.

Architecture: Multi-Topic Research

Topic Fusion of AI Agents & MCP

<img width="600" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/parallel-agent.png" alt="Parallel Agent" />

Example: Parallel Multi-Topic Research

Let's build a system with four agents:

    Researcher on Decentralized Identity & Access Control=tech1_researcher
    
    Researcher on Verifiable Credentials for Model Provenance=tech2_researcher
    
    Researcher on Smart Contract-Driven Workflow Orchestration=tech3_researcher

    Researcher on On-Chain Audit Trails for Every Interaction=tech4_researcher
    
    Researcher on Tokenized API Access & Monetization=tech5_researcher

    Researcher on Agent-to-Agent Communication Protocols=tech6_researcher
    
    Researcher on AI Agent Reputation & Trust Scoring=tech7_researcher
    
    Researcher on Automated Data Lineage Tracking=tech8_researcher

    Researcher on Intelligent Resource Allocation & Management=tech9_researcher
    
    Researcher on Intelligent Resource Allocation & Management=tech10_researcher
    
    Researcher on Decentralized Model Registry & Versioning=tech11_researcher

    Researcher on Addressing Data Exfiltration Risks=tech12_researcher

    Researcher on API Security and Access Management=tech13_researcher
    
    Researcher on Preventing Malicious Agent Behavior=tech14_researcher
    

In [18]:
#Researcher on Decentralized Identity & Access Control: Focuses on Decentralized Identity & Access Control.
tech1_researcher = Agent(
    name="tech1Researcher", #Researcher on Decentralized Identity & Access Control=tech1_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how to implement self-sovereign identities for AI agents and users on Algorand.And how to allow for granular, verifiable, 
    and secure management of permissions, such that it will ensure that only authorized entities can access specific tools, models, or data. 
    Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech1_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech1_researcher created.")

✅ tech1_researcher created.


In [19]:
#Researcher on Verifiable Credentials for Model Provenance: Focuses on Verifiable Credentials for Model Provenance.
tech2_researcher = Agent(
    name="tech2Researcher", #Researcher on Verifiable Credentials for Model Provenance=tech2_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how to utilize Algorand Standard Assets (ASAs) to represent attestations about 
    AI models. The research should includes details about training data sources, performance metrics,
    ethical compliance, and origin, creating a transparent and trustworthy record of model
    lineage.Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech2_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech2_researcher created.")

✅ tech2_researcher created.


In [20]:
#Researcher on Smart Contract-Driven Workflow Orchestration: Focuses on Smart Contract-Driven Workflow Orchestration.
tech3_researcher = Agent(
    name="tech3Researcher", #Researcher on Smart Contract-Driven Workflow Orchestration=tech3_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how improve Model Context Protocol(MCP) and Algorand blockchain
    to automate complex integration sequences and data flows between various tools and AI 
    models using Algorand smart contracts, so as to ensures predictablity, reliablity,and 
    auditable execution of multi-step processes.Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech3_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech3_researcher created.")

✅ tech3_researcher created.


In [21]:
#Researcher on On-Chain Audit Trails for Every Interaction: Focuses on On-Chain Audit Trails for Every Interaction.
tech4_researcher = Agent(
    name="tech4Researcher", #Researcher on On-Chain Audit Trails for Every Interaction=tech4_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on Algorand blockchain and Model Context Protocol(MCP) 
    can do On-Chain Audit Trails for Every Interaction by recording of all AI agent actions, tool usage, data access,
    and model inferences directly on the immutable Algorand ledger. Make sure your 
    research on On-Chain Audit Trails for Every Interaction on the Algorand blockchain
    integration with the model context protocol(MCP) provides a complete, 
    tamper-proof history for accountability, compliance, and debugging.
    Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech4_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech4_researcher created.")

✅ tech4_researcher created.


In [22]:
#Researcher on Tokenized API Access & Monetization: Focuses on Tokenized API Access & Monetization.
tech5_researcher = Agent(
    name="tech5Researcher", #Researcher on Tokenized API Access & Monetization=tech5_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how tokenized API Access & Monetization in the 
    fusion of AI Agents, Algorand Blockchain and Model Context Protocol(MCP) can carry out
    the representation of access rights to proprietary APIs as Algorand Standard Assets 
    (ASAs). That will enables secure,auditable, and potentially monetizable access control
    mechanisms, and how it can allowed for controlled distribution and revenue generation.
    Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech5_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech5_researcher created.")

✅ tech5_researcher created.


In [23]:
#Researcher on Agent-to-Agent Communication Protocols: Focuses on Agent-to-Agent Communication Protocols.
tech6_researcher = Agent(
    name="tech6Researcher", #Researcher on Agent-to-Agent Communication Protocols=tech6_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how Establish secure and standardized communication channels 
    between AI agents through Model Context Protocol(MCP) on Algorand Blochchain.
    Let this protocols be reinforced by Algorand transactions, ensuring the integrity and authenticity of inter-agent messages.Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech6_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech6_researcher created.")

✅ tech6_researcher created.


In [24]:
#Researcher on AI Agent Reputation & Trust Scoring: Focuses on AI Agent Reputation & Trust Scoring.
tech7_researcher = Agent(
    name="tech7Researcher", #Researcher on AI Agent Reputation & Trust Scoring=tech7_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on to develop decentralized reputation systems for AI agents 
    on the Algorand blockchain fused together with Model Context Protocol . 
    This should allows the ecosystem to assess the reliability and trustworthiness of 
    agents based on their historical performance and interactions,
    fostering a more dependable network..Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech7_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech7_researcher created.")

✅ tech7_researcher created.


In [25]:
#Researcher on Automated Data Lineage Tracking: Focuses on Automated Data Lineage Tracking.
tech8_researcher = Agent(
    name="tech8Researcher", #Researcher on Automated Data Lineage Tracking=tech8_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how to Leverage smart contracts to improve 
    Model Context Protocol(MCP) to automatically record the origin, transformations, 
    and usage of data throughout its lifecycle.And how this should ensures clear 
    understanding of data provenance and facilitates compliance with data governance 
    policies.Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech8_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech8_researcher created.")

✅ tech8_researcher created.


In [26]:
#Researcher on Intelligent Resource Allocation & Management: Focuses on Intelligent Resource Allocation & Management.
tech9_researcher = Agent(
    name="tech9Researcher", #Researcher on Intelligent Resource Allocation & Management=tech9_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how to Use AI agents to dynamically allocate
    computational resources to models based on demand and priority, recorded 
    on-chain,considering the improvement of Model Context Protocol(MCP)on Algorand Blockchain
    .Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech9_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech9_researcher created.")

✅ tech9_researcher created.


In [27]:
#Researcher on Intelligent Resource Allocation & Management: Focuses on Intelligent Resource Allocation & Management.
tech10_researcher = Agent(
    name="tech10Researcher", #Researcher on Intelligent Resource Allocation & Management=tech10_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how to Empower AI agents to dynamically allocate 
    computational resources(e.g., processing power, storage) to models based on 
    real-time demand, priority, and cost-effectiveness, with key
    allocation decisions recorded on-chain for transparency with the intension of 
    improving the risk mitigation of Model Context Protocol on 
    Algoran Blockchani.Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech10_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech10_researcher created.")

✅ tech10_researcher created.


In [28]:
#Researcher on Decentralized Model Registry & Versioning: Focuses on Decentralized Model Registry & Versioning.
tech11_researcher = Agent(
    name="tech11Researcher", #Researcher on Decentralized Model Registry & Versioning=tech11_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how to Store metadata, version history, and 
    performance characteristics of AI models in a decentralized and verifiable manner 
    on Algorand.Inorder to ensures that organizations are always referencing the
    correct and approved versions of their models through Model Context Protocol(MCP).
    Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech11_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech11_researcher created.")

✅ tech11_researcher created.


In [29]:
#Researcher on Addressing Data Exfiltration Risks: Focuses on Addressing Data Exfiltration Risks.
tech12_researcher = Agent(
    name="tech12Researcher", #Researcher on Addressing Data Exfiltration Risks=tech12_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how Algorand's immutable ledger, coupled with granular access controls enforced by smart contracts,
    can significantly limits the potential for data exposure. And how AI agents can be programmed to monitor data access patterns, 
    flagging any suspicious activity in real-time, while smart contracts ensure that data is only shared with explicitly 
    authorized parties.Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech12_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech12_researcher created.")

✅ tech12_researcher created.


In [30]:
#Researcher on API Security and Access Management: Focuses on API Security and Access Management.
tech13_researcher = Agent(
    name="tech13Researcher", #Researcher on API Security and Access Management=tech13_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how Algorand Standard Assets (ASAs) can function as secure, auditable, and easily revocable tokens
    for API access. This must moves beyond traditional, often vulnerable, API key management, providing a transparent and controlled 
    mechanism that will prevents unauthorized use and simplifies access revocation.Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech13_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech13_researcher created.")

✅ tech13_researcher created.


In [31]:
#Researcher on Preventing Malicious Agent Behavior: Focuses on Preventing Malicious Agent Behavior.
tech14_researcher = Agent(
    name="tech14Researcher", #Researcher on Preventing Malicious Agent Behavior=tech14_researcher
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Research on how the inherent immutability of the Algorand blockchain acts as a powerful deterrent against
    malicious actions, and how any recorded transaction or state change cannot be altered. Furthermore, give details on how
    AI agents can be trained to detect and flag anomalous or suspicious patterns of behavior within the network, and these 
    findings can be immutably recorded on-chain, creating a verifiable record for analysis and action.Keep it very concise (100 words).""",
    tools=[google_search],
    output_key="tech14_research",  # The result of this agent will be stored in the session state with this key.
)

print("✅ tech14_researcher created.")

✅ tech14_researcher created.


In [32]:
# The AggregatorAgent runs *after* the parallel step to synthesize the results.
aggregator_agent = Agent(
    name="AggregatorAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    # It uses placeholders to inject the outputs from the parallel agents, which are now in the session state.
    instruction="""The integration of AI agents with Algorand's blockchain technology 
    provides a fertile ground for enhancing your existing Model Context Protocol (MCP). 
    Here are fourteen key ways this synergy can be leveraged by Combining these
    fourteen research findings into a single executive summary:

    Researcher on Decentralized Identity & Access Control=tech1_researcher:
    {tech1_research}
    
    Researcher on Verifiable Credentials for Model Provenance=tech2_researcher:
    {tech2_research}
    
    Researcher on Smart Contract-Driven Workflow Orchestration=tech3_researcher:
    {tech3_research}

    Researcher on On-Chain Audit Trails for Every Interaction=tech4_researcher:
    {tech4_research}**
    
    Researcher on Tokenized API Access & Monetization=tech5_researcher:
    {tech5_research}

    Researcher on Agent-to-Agent Communication Protocols=tech6_researcher:
    {tech6_research}
    
    Researcher on AI Agent Reputation & Trust Scoring=tech7_researcher:
    {tech7_research}

    Researcher on Automated Data Lineage Tracking=tech8_researcher:
    {tech8_research}

    Researcher on Intelligent Resource Allocation & Management=tech9_researcher:
    {tech9_research}
    
    Researcher on Intelligent Resource Allocation & Management=tech10_researcher:
    {tech10_research}
    
    Researcher on Decentralized Model Registry & Versioning=tech11_researcher:
    {tech11_research}

    Researcher on Addressing Data Exfiltration Risks=tech12_researcher:
    {tech12_research}

    Researcher on API Security and Access Management=tech13_researcher:
    {tech13_research}

    Researcher on Preventing Malicious Agent Behavior=tech14_researcher:
    {tech14_research}
    
    Your summary should highlight common themes, surprising connections, and the most important key takeaways from all fourteen reports. The final summary should be around 200 words.""",
    output_key="executive_summary",  # This will be the final output of the entire system.
)

print("✅ aggregator_agent created.")

✅ aggregator_agent created.


Then we bring the agents together under a parallel agent, which is itself nested inside of a sequential agent.

This design ensures that the research agents run first in parallel, then once all of their research is complete, the aggregator agent brings together all of the research findings into a single report:

In [33]:
# The ParallelAgent runs all its sub-agents simultaneously.
parallel_research_team = ParallelAgent(
    name="ParallelResearchTeam",
    sub_agents=[tech1_researcher,tech2_researcher,tech3_researcher,tech4_researcher,tech5_researcher,tech6_researcher,tech7_researcher,
               tech8_researcher,tech9_researcher,tech10_researcher,tech11_researcher,tech12_researcher,tech13_researcher,tech14_researcher],
)

# This SequentialAgent defines the high-level workflow: run the parallel team first, then run the aggregator.
root_agent = SequentialAgent(
    name="ResearchSystem",
    sub_agents=[parallel_research_team, aggregator_agent],
)

print("✅ Parallel and Sequential Agents created.")

✅ Parallel and Sequential Agents created.


In [34]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    """Run the daily executive briefing on tech1_researcher,tech2_researcher,tech3_researcher,tech4_researcher,tech5_researcher,
       tech6_researcher,tech7_researcher,tech8_researcher,tech9_researcher,tech10_researcher,tech11_researcher,tech12_researcher,
       tech13_researcher,tech14_researcher.

       Finish it with an wholesome conculosion
    """
)


 ### Created new session: debug_session_id

User > Run the daily executive briefing on tech1_researcher,tech2_researcher,tech3_researcher,tech4_researcher,tech5_researcher,
       tech6_researcher,tech7_researcher,tech8_researcher,tech9_researcher,tech10_researcher,tech11_researcher,tech12_researcher,
       tech13_researcher,tech14_researcher.

       Finish it with an wholesome conculosion
    
tech1Researcher > I'm sorry, but I cannot fulfill this request. I do not have the capability to run daily executive briefings or access specific research agents like "tech1_researcher" through "tech14_researcher." My purpose is to provide information and complete tasks based on the data I was trained on and can access through search.
tech11Researcher > I cannot fulfill this request. I do not have the capability to run daily executive briefings for multiple agents or access their individual research. My function is to respond to your direct queries.
tech2Researcher > Algorand Standard Assets (

Loop Workflows - The Refinement Cycle

The Problem: One-Shot Quality

All the workflows we've seen so far run from start to finish. The SequentialAgent and ParallelAgent produce their final output and then stop. This 'one-shot' approach isn't good for tasks that require refinement and quality control. What if the first draft of our story is bad? We have no way to review it and ask for a rewrite.

The Solution: Iterative Refinement

When a task needs to be improved through cycles of feedback and revision, you can use a LoopAgent. A LoopAgent runs a set of sub-agents repeatedly until a specific condition is met or a maximum number of iterations is reached. This creates a refinement cycle, allowing the agent system to improve its own work over and over.

Use Loop when: Iterative improvement is needed, quality refinement matters, or you need repeated cycles.

To learn more, check out the documentation related to loop agents in ADK.

Architecture: Story Writing & Critique Loop

<img width="250" src="https://storage.googleapis.com/github-repo/kaggle-5days-ai/day1/loop-agent.png" alt="Loop Agent" />

Iterative Story Refinement

Let's build a system with two agents:

    Writer Agent - Writes a draft from the root agent report
    Critic Agent - Reviews and critiques the root agent report to suggest improvements


In [35]:
# This agent runs ONCE at the beginning to create the first draft.
initial_writer_agent = Agent(
    name="InitialWriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""Based on the daily executive briefing on tech1_researcher,tech2_researcher,tech3_researcher,
    tech4_researcher,tech5_researcher,tech6_researcher,tech7_researcher,tech8_researcher,tech9_researcher,tech10_researcher,
    tech11_researcher,tech12_researcher,tech13_researcher,tech14_researcher
       
), write the first draft from the root agent report (around 1350-1400 words).
    Output only the daily executive briefing text, with no introduction or explanation.""",
    output_key="current_dailyExecutiveBriefing",  # Stores the first draft in the state.
)

print("✅ initial_writer_agent created.")

✅ initial_writer_agent created.


In [36]:
# This agent's only job is to provide feedback or the approval signal. It has no tools.
critic_agent = Agent(
    name="CriticAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a constructive daily executive briefing critic. Review the daily executive briefing provided below.
    dailyExecutiveBriefing: {current_dailyExecutiveBriefing}
    
    Evaluate the executive briefing plot, characters, and pacing.
    - If the daily executive briefing is well-written and complete, you MUST respond with the exact phrase: "APPROVED"
    - Otherwise, provide 2-3 specific, actionable suggestions for improvement.""",
    output_key="critique",  # Stores the feedback in the state.
)

print("✅ critic_agent created.")

✅ critic_agent created.


Now, we need a way for the loop to actually stop based on the critic's feedback. Because the LoopAgent itself doesn't automatically know that "APPROVED" means "stop."

We need an agent to give it an explicit signal to terminate the loop.

We do this in two parts:

    A simple Python function that the LoopAgent understands as an "exit" signal.
    An agent that can call that function when the right condition is met.

First, you'll define the exit_loop function:

In [37]:
# This is the function that the RefinerAgent will call to exit the loop.
def exit_loop():
    """Call this function ONLY when the critique is 'APPROVED', indicating the daily executive briefing is finished and no more changes are needed."""
    return {"status": "approved", "message": "dailyExecutiveBriefing approved. Exiting refinement loop."}


print("✅ exit_loop function created.")

✅ exit_loop function created.


To let an agent call this Python function, we wrap it in a FunctionTool. Then, we create a RefinerAgent that has this tool.

👉 Notice its instructions: this agent is the "brain" of the loop. It reads the {critique} from the CriticAgent and decides whether to (1) call the exit_loop tool or (2) rewrite the daily executive briefing.


In [38]:
# This agent refines the story based on critique OR calls the exit_loop function.
refiner_agent = Agent(
    name="RefinerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction="""You are a daily executive briefing refiner. You have a daily executive briefing draft and critique.
    
    daily executive briefing Draft: {current_dailyExecutiveBriefing}
    Critique: {critique}
    
    Your task is to analyze the critique.
    - IF the critique is EXACTLY "APPROVED", you MUST call the `exit_loop` function and nothing else.
    - OTHERWISE, rewrite the daily executive briefing draft to fully incorporate the feedback from the critique.""",
    output_key="current_dailyExecutiveBriefing",  # It overwrites the story with the new, refined version.
    tools=[
        FunctionTool(exit_loop)
    ],  # The tool is now correctly initialized with the function reference.
)

print("✅ refiner_agent created.")

✅ refiner_agent created.


Now let us bring the agents together under a loop agent, which is itself nested inside of a sequential agent.

This design ensures that the system first produces an initial daily executive briefing draft, then the refinement loop runs up to the specified number of max_iterations:

In [39]:
# The LoopAgent contains the agents that will run repeatedly: Critic -> Refiner.
dailyexecutivebriefing_refinement_loop = LoopAgent(
    name="DailyExecutiveBriefingRefinementLoop",
    sub_agents=[critic_agent, refiner_agent],
    max_iterations=2,  # Prevents infinite loops
)

# The root agent is a SequentialAgent that defines the overall workflow: Initial Write -> Refinement Loop.
root_agent = SequentialAgent(
    name="DailyExecutiveBriefingPipeline",
    sub_agents=[initial_writer_agent, dailyexecutivebriefing_refinement_loop],
)

print("✅ Loop and Sequential Agents created.")

✅ Loop and Sequential Agents created.


In [40]:
runner = InMemoryRunner(agent=root_agent)
response = await runner.run_debug(
    """Write a daily executive briefing reporton tech1_researcher,tech2_researcher,tech3_researcher,
    tech4_researcher,tech5_researcher,tech6_researcher,tech7_researcher,tech8_researcher,tech9_researcher,tech10_researcher,
    tech11_researcher,tech12_researcher,tech13_researcher,tech14_researcher.
    
    """
)


 ### Created new session: debug_session_id

User > Write a daily executive briefing reporton tech1_researcher,tech2_researcher,tech3_researcher,
    tech4_researcher,tech5_researcher,tech6_researcher,tech7_researcher,tech8_researcher,tech9_researcher,tech10_researcher,
    tech11_researcher,tech12_researcher,tech13_researcher,tech14_researcher.
    
    
InitialWriterAgent > ## Daily Executive Briefing: Technology Research

**Date:** October 26, 2023

**Prepared For:** Executive Leadership

**Prepared By:** InitialWriterAgent (Root Agent Report)

**Subject:** Synthesis of Daily Technology Research Insights

This briefing provides a consolidated overview of the key findings and developments emerging from our dedicated technology research team (tech1-tech14) over the past 24 hours. The research spans a broad spectrum of technological advancements, market trends, and potential strategic implications, offering a snapshot of the evolving landscape relevant to our organization.

---

**I. E

ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7ddc0b094490>
ERROR:asyncio:Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7ddc0b09cbd0>
